# PaaS Agent 测试用例

本 Notebook 用于验证迁移到 LangGraph Platform 后的 Agent 行为。

运行前请确保：
1. 已在 `paas_project` 目录下。
2. `langgraph.json` 配置正确。
3. 如需 HTTP 测试，先启动 `langgraph dev --config langgraph.json --port 8123 --allow-blocking --no-browser`。

## 1. 图结构与导入测试

In [9]:
from paas_core.agent.langgraph.main_graph import graph, build_main_graph

# 1.1 模块级 graph 实例存在且为 CompiledGraph
assert graph is not None, "模块级 graph 实例不存在"
assert hasattr(graph, "invoke"), "graph 必须支持 invoke"
assert hasattr(graph, "astream_events"), "graph 必须支持 astream_events"

# 1.2 build_main_graph 每次返回新的编译后图
g2 = build_main_graph()
assert g2 is not graph, "build_main_graph 应返回新的实例"
print("✅ 图导入与结构测试通过")

✅ 图导入与结构测试通过


In [10]:
# 1.3 检查主图节点与边
expected_nodes = {"react_0", "react_1", "parse_task_status"}
actual_nodes = set(graph.get_graph().nodes.keys())
missing = expected_nodes - actual_nodes
assert not missing, f"缺少节点: {missing}"

edges = graph.get_graph().edges
edge_pairs = {(e.source, e.target) for e in edges}
assert ("__start__", "react_0") in edge_pairs, "缺少 START -> react_0 边"
assert ("react_0", "react_1") in edge_pairs, "缺少 react_0 -> react_1 边"
assert ("react_1", "parse_task_status") in edge_pairs, "缺少 react_1 -> parse_task_status 边"
print("✅ 节点与边检查通过")

✅ 节点与边检查通过


## 2. 同步 invoke 测试

In [11]:
from langchain_core.messages import HumanMessage

# 2.1 简单计算任务
input_state = {
    "messages": [HumanMessage(content="计算 2 + 3")],
    "task_status": "incomplete",
}
result = graph.invoke(input_state, {"recursion_limit": 10})

messages = result.get("messages", [])
assert len(messages) > 1, "应产生多条消息（用户 + AI + 工具 + AI ...）"

final_content = messages[-1].content
assert "5" in final_content, f"最终答案应包含 5，实际为: {final_content}"
print("✅ invoke 计算测试通过")
print("最终答案:", final_content.strip())

✅ invoke 计算测试通过
最终答案: 任务状态：已完成
最终回答：2 + 3 = 5


In [12]:
# 2.2 task_status 最终应被解析为 completed
assert result.get("task_status") == "completed", f"任务状态应为 completed，实际为: {result.get('task_status')}"
print("✅ task_status 状态检查通过")

✅ task_status 状态检查通过


## 3. 流式事件测试（astream_events v2）

In [13]:
import asyncio

async def stream_test():
    events = []
    async for event in graph.astream_events(
        {
            "messages": [HumanMessage(content="计算 3 + 4")],
            "task_status": "incomplete",
        },
        {"recursion_limit": 10},
        version="v2",
    ):
        events.append(event)
        if event["event"] in {"on_chat_model_stream", "on_tool_start", "on_tool_end"}:
            print(event["event"], event.get("name", ""))
    return events

events = await stream_test()

event_names = {e["event"] for e in events}
assert "on_chain_start" in event_names, "缺少 on_chain_start"
assert "on_chain_end" in event_names, "缺少 on_chain_end"
assert "on_chat_model_stream" in event_names, "缺少 on_chat_model_stream"
assert "on_tool_start" in event_names, "缺少 on_tool_start"
assert "on_tool_end" in event_names, "缺少 on_tool_end"
print("\n✅ 流式事件测试通过，事件种类:", event_names)

on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_model_stream ChatOpenAI
on_chat_

## 4. LangGraph Platform HTTP API 测试

In [14]:
import requests

BASE_URL = "http://localhost:8123"
ASSISTANT_ID = "paas-agent"

# 4.1 服务可达性检查
r = requests.get(f"{BASE_URL}/assistants")
assert r.status_code == 200, f"无法访问 {BASE_URL}/assistants: {r.status_code}"
print("✅ Platform API 可达")

AssertionError: 无法访问 http://localhost:8123/assistants: 405

In [15]:
# 4.2 创建 thread
r = requests.post(f"{BASE_URL}/threads", json={})
assert r.status_code == 200, f"创建 thread 失败: {r.status_code} {r.text}"
thread = r.json()
thread_id = thread["thread_id"]
print(f"✅ Thread 创建成功: {thread_id}")

✅ Thread 创建成功: 019f8c92-fe15-71f1-b261-438a5b4b4c48


In [16]:
# 4.3 流式运行 thread
payload = {
    "assistant_id": ASSISTANT_ID,
    "input": {
        "messages": [{"type": "human", "content": "计算 2 + 3"}],
        "task_status": "incomplete",
    },
    "stream_mode": ["events", "values"],
}

seen_events = set()
final_answer = None

with requests.post(
    f"{BASE_URL}/threads/{thread_id}/runs/stream",
    json=payload,
    stream=True,
) as r:
    assert r.status_code == 200, f"流式运行失败: {r.status_code} {r.text}"
    for line in r.iter_lines():
        if not line:
            continue
        text = line.decode("utf-8")
        if text.startswith("event:"):
            event_type = text.split(":", 1)[1].strip()
            seen_events.add(event_type)
        elif text.startswith("data:"):
            data = text.split(":", 1)[1].strip()
            try:
                import json
                payload_data = json.loads(data)
                if payload_data.get("event") == "on_chain_end" and payload_data.get("name") == "paas-agent":
                    msgs = payload_data.get("data", {}).get("output", {}).get("messages", [])
                    if msgs:
                        final_answer = msgs[-1].get("content")
            except Exception:
                pass

assert "events" in seen_events, "SSE 应包含 events 事件"
assert "values" in seen_events, "SSE 应包含 values 事件"
assert final_answer and "5" in final_answer, f"最终答案应包含 5，实际: {final_answer}"
print("✅ Platform 流式运行测试通过")
print("最终答案:", final_answer.strip())

✅ Platform 流式运行测试通过
最终答案: 任务状态：已完成
最终回答：2 + 3 = 5


## 5. 系统服务导入测试

In [17]:
# 验证移除 /admin/agent/* 后 system_server 仍可正常导入
from paas_core.server.system.system_server import app

routes = {route.path for route in app.routes}
print("已注册路由:", sorted(routes)[:20])

# 不应再存在旧的 agent 端点
agent_routes = {p for p in routes if "admin/agent" in p}
assert not agent_routes, f"残留 admin/agent 路由: {agent_routes}"
print("✅ 系统服务导入与路由清理测试通过")

ImportError: cannot import name 'app' from 'paas_core.server.system.system_server' (/Users/lanzhengpeng/develop/thesis/paas_project/paas_core/server/system/system_server.py)

## 6. 取消运行测试（可选，需运行中任务）

In [ ]:
# 6.1 发起一个可能较长的任务并立即取消
r = requests.post(f"{BASE_URL}/threads", json={})
assert r.status_code == 200
cancel_thread_id = r.json()["thread_id"]

payload = {
    "assistant_id": ASSISTANT_ID,
    "input": {
        "messages": [{"type": "human", "content": "请详细介绍 LangGraph 的所有功能"}],
        "task_status": "incomplete",
    },
    "stream_mode": ["events", "values"],
}

# 先创建 run（不流式读取）
r = requests.post(f"{BASE_URL}/threads/{cancel_thread_id}/runs", json=payload)
assert r.status_code in {200, 202}, f"创建 run 失败: {r.status_code} {r.text}"
run_id = r.json()["run_id"]

# 立即取消
r = requests.post(f"{BASE_URL}/threads/{cancel_thread_id}/runs/{run_id}/cancel")
assert r.status_code in {200, 202, 204}, f"取消 run 失败: {r.status_code} {r.text}"
print(f"✅ Run 取消测试通过: {run_id}")

✅ Run 取消测试通过: 019f8c93-3f4a-7782-8e46-4860c6069103


: 